# HACE Hedge Detection Sanity Check

Version 1 uses a deterministic, lexicon-derived hedge confidence score. It is not a calibrated ML probability.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.hedging.detector import HedgeDetector

detector = HedgeDetector()
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
test_sentences = [
    'The company reported record revenue.',
    'The company may report stronger earnings.',
    'The results suggest that revenue could increase.',
    'Depending on market conditions, earnings could improve.',
    'The word maybe should not match the cue may.',
]

for text in test_sentences:
    print(text)
    print(detector.detect(text).to_dict())


In [ ]:
fiqa_path = PROJECT_ROOT / 'data' / 'raw' / 'fiqa' / 'train.csv'
fiqa = pd.read_csv(fiqa_path).copy()
results = fiqa['sentence'].map(detector.detect)
fiqa['hedge_flag'] = results.map(lambda item: item.hedge_flag)
fiqa['hedge_probability'] = results.map(lambda item: item.hedge_probability)
fiqa['hedge_count'] = results.map(lambda item: item.hedge_count)
fiqa['detected_terms'] = results.map(lambda item: item.detected_terms)

print('Total sentences:', len(fiqa))
print('Hedged sentences:', int(fiqa['hedge_flag'].sum()))
print('Non-hedged sentences:', int((~fiqa['hedge_flag']).sum()))
print('Hedging percentage:', round(fiqa['hedge_flag'].mean() * 100, 2))
print('Average hedge probability:', round(fiqa['hedge_probability'].mean(), 4))
print('Average hedge count:', round(fiqa['hedge_count'].mean(), 4))

display(fiqa.loc[fiqa['hedge_flag'], ['sentence', 'hedge_probability', 'hedge_count', 'detected_terms']].head(20))
display(fiqa.loc[~fiqa['hedge_flag'], ['sentence', 'hedge_probability', 'hedge_count', 'detected_terms']].head(20))

output_path = PROJECT_ROOT / 'data' / 'interim' / 'fiqa_hedging.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
fiqa.to_csv(output_path, index=False)
print(f'Saved enriched data to: {output_path}')